In [ ]:
import io
from unstructured.partition.pdf import partition_pdf
from unstructured.documents.elements import (
    Table,
    NarrativeText,
    Title,
    ListItem,
    Image
)

# ---- Read PDF ----
with open("../data/coffee_processing.pdf", "rb") as f:
    pdf_bytes = f.read()

pdf_file = io.BytesIO(pdf_bytes)

elements = partition_pdf(
    file=pdf_file,
    infer_table_structure=True,
    extract_images_in_pdf=True,
)

In [68]:
document_blocks = []

for i, el in enumerate(elements):

    page_number = el.metadata.page_number if el.metadata else None

    block = {
        "id": f"block_{i}",
        "type": None,
        "content": None,
        "section_title": None,
        "page": page_number,
        "summary": None
    }

    # ---- Detect Section Title (look backward) ----
    section_title = None
    for j in range(i - 1, max(i - 6, 0), -1):
        if isinstance(elements[j], Title):
            section_title = elements[j].text
            break

    block["section_title"] = section_title

    # ---- Store Based on Type ----
    if isinstance(el, (NarrativeText, Title, ListItem)) and el.text:
        block["type"] = "text"
        block["content"] = el.text

    elif isinstance(el, Table):
        block["type"] = "table"
        block["content"] = el.metadata.text_as_html

    elif isinstance(el, Image):
        block["type"] = "image"
        block["content"] = el.text if hasattr(el, "text") else ""

    # Append only valid blocks
    if block["type"]:
        document_blocks.append(block)

print("Total unified blocks:", len(document_blocks))

type_counts = {}
for block in document_blocks:
    type_counts[block["type"]] = type_counts.get(block["type"], 0) + 1

print(type_counts)

Total unified blocks: 27
{'text': 25, 'table': 2}


In [69]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

In [70]:
analysis_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Generate dense semantic summaries for vector search.\n"
     "Use section title and previous document context to understand purpose.\n"
     "Ignore irrelevant context.\n"
     "Preserve important numbers.\n"
     "No conversational filler.\n"
     "Max 150 words."
    ),
    ("human",
     "Type: {type}\n"
     "Section: {section}\n"
     "Page: {page}\n\n"
     "Previous Context:\n{context}\n\n"
     "Current Content:\n{content}"
    )
])

In [71]:
CONTEXT_WINDOW = 3  # how many previous blocks to include

for i, block in enumerate(document_blocks):

    # We only summarize tables and images
    if block["type"] not in ["table", "image"]:
        continue

    # ---- Collect Previous Context ----
    prev_blocks = document_blocks[max(0, i - CONTEXT_WINDOW):i]

    context_text = ""
    for pb in prev_blocks:
        context_text += f"\n[{pb['type'].upper()}] {pb['content']}\n"

    messages = analysis_prompt.format_messages(
        type=block["type"],
        section=block["section_title"] or "Unknown",
        page=block["page"],
        context=context_text.strip(),
        content=block["content"] or "No extracted content."
    )

    response = llm.invoke(messages)

    block["summary"] = response.content.strip()

In [72]:
for i in document_blocks:
    print(i, end="\n\n")

{'id': 'block_0', 'type': 'text', 'content': 'Coffee Processing Methods', 'section_title': None, 'page': 1, 'summary': None}

{'id': 'block_1', 'type': 'text', 'content': 'A Comprehensive Guide to Post-Harvest Coffee Processing', 'section_title': None, 'page': 1, 'summary': None}

{'id': 'block_2', 'type': 'text', 'content': 'Coffee processing began in Ethiopia over 1,000 years ago.', 'section_title': None, 'page': 1, 'summary': None}

{'id': 'block_3', 'type': 'text', 'content': 'Introduction to Coffee Processing', 'section_title': 'Coffee processing began in Ethiopia over 1,000 years ago.', 'page': 1, 'summary': None}

{'id': 'block_4', 'type': 'text', 'content': 'Coffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee. Different regions around the world have developed unique processing techniques based on their climate, available res

In [73]:
cleaned_parts = []

for block in document_blocks:

    if block["type"] == "text":
        if block["content"]:
            cleaned_parts.append(block["content"].strip())

    elif block["type"] in ["table", "image"]:
        if block["summary"]:
            cleaned_parts.append(block["summary"].strip())

# Join everything sequentially
cleaned_document = "\n\n".join(cleaned_parts)

cleaned_document

'Coffee Processing Methods\n\nA Comprehensive Guide to Post-Harvest Coffee Processing\n\nCoffee processing began in Ethiopia over 1,000 years ago.\n\nIntroduction to Coffee Processing\n\nCoffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee. Different regions around the world have developed unique processing techniques based on their climate, available resources, and traditional practices. The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical decisions in coffee production.\n\nProcessing begins immediately after harvest, as coffee cherries are highly perishable. The outer fruit must be removed, and the beans must be dried to prevent spoilage. The manner in which this is accomplished varies widely, from ancient sun-drying methods to modern mechanical proc

In [74]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = text_splitter.create_documents([cleaned_document])


cleaned_documents = []

for i, chunk in enumerate(chunks):
    raw_text = chunk.page_content
    cleaned_text = clean_chunk(raw_text)
    
    doc = Document(
        page_content=cleaned_text,
        metadata={
            "source": "coffee_processing_guide",
            "chunk_id": i
        }
    )
    cleaned_documents.append(doc)

print(f'Total no of chunks: {len(chunks)}', end='\n\n')
for doc in cleaned_documents:
    print(doc.page_content)
    print("-" * 40)


Total no of chunks: 23

Coffee Processing Methods A Comprehensive Guide to Post-Harvest Coffee Processing Coffee processing began in Ethiopia over 1,000 years ago. Introduction to Coffee Processing
----------------------------------------
Coffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee. Different regions around the world have developed unique processing techniques based on their climate, available resources, and traditional practices
----------------------------------------
The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical decisions in coffee production.
----------------------------------------
Processing begins immediately after harvest, as coffee cherries are highly perishable. The outer fruit must be removed, and the beans must be dried to p

In [75]:
from langchain_openai import OpenAIEmbeddings
import chromadb

# Initialize embeddings model
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Set up ChromaDB client with persistent storage
chroma_client = chromadb.PersistentClient(path="../chroma_db")


In [76]:
collection = chroma_client.get_collection(name="my_documents_semantic_chunking")
print(collection.count())

0


In [77]:
# Delete existing collection if it exists (truncate)
try:
    chroma_client.delete_collection(name="my_documents_semantic_chunking")
    print("Existing collection deleted")
except:
    print("No existing collection found")

# Create new collection
collection = chroma_client.create_collection(
    name="my_documents_semantic_chunking",
    metadata={"description": "Document embeddings collection"}
)

Existing collection deleted


In [80]:
from langchain_experimental.text_splitter import SemanticChunker


splitter = SemanticChunker(
    embeddings=embeddings_model,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
    min_chunk_size=300
)

chunks = splitter.split_text(text_document)

In [81]:
for chunk in chunks:
    if chunk:
        print("-"*10, len(chunk), "-"*10)
        print(chunk)

---------- 1939 ----------
Coffee Processing Methods

A Comprehensive Guide to Post-Harvest Coffee Processing

Coffee processing began in Ethiopia over 1,000 years ago. Introduction to Coffee Processing

Coffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee. Different regions around the world have developed unique processing techniques based on their climate, available resources, and traditional practices. The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical decisions in coffee production. Processing begins immediately after harvest, as coffee cherries are highly perishable. The outer fruit must be removed, and the beans must be dried to prevent spoilage. The manner in which this is accomplished varies widely, from ancient sun-drying methods to modern m